## Ray Foundations  
In this module we will learn the following:  
- Tasks
- Actors
- Object Store

In [4]:
## Ray Core
import ray
import torch
import os
import time
from pprint import pprint
import numpy as np
from typing import Any

In [2]:
context = ray.init(
    ignore_reinit_error=True, 
    include_dashboard=True, 
    dashboard_host="0.0.0.0"
)
print(context.dashboard_url)

2026-04-28 22:55:11,274	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at http://172.28.165.76:8265 


172.28.165.76:8265


In [3]:
torch.cuda.is_available()

True

### 1. Ray Tasks  
#### What is a Ray Task?
In normal Python, a function runs synchronously — the caller waits until it finishes. A Ray task runs asynchronously on a separate worker process, potentially on a different CPU core or machine entirely. Your main program keeps going while the task runs. The mental model is something like this:  
```mermaid
Normal Python:          Ray Task:
─────────────           ─────────────────────────
call function()    →    submit task → get ObjectRef (immediately)
wait...            →    do other work...
get result         →    ray.get(ref) → block only when you need result
```

**Step 1 — Your first task**  
Run this and observe the output carefully:

In [4]:
@ray.remote
def add(a, b):
    return a + b

In [5]:
# .remote() submits the task — returns immediately
ref = add.remote(3, 7)

print(type(ref))   # ray.ObjectRef — NOT the result yet
print(ref)         # ObjectRef(abc123...)

# ray.get() blocks until result is ready
result = ray.get(ref)
print(result)      # 10

2026-04-27 11:36:09,748	INFO worker.py:2012 -- Started a local Ray instance.


<class 'ray.ObjectRef'>
ObjectRef(c8ef45ccd0112571ffffffffffffffffffffffff0100000001000000)
10


/mnt/c/Users/mrinm/Downloads/MyTraining/DeepLearning/Pytorch_and_Ray/Ray/.venv/lib/python3.13/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


```mermaid
Your Script (main process)              Ray Worker Process
──────────────────────────              ──────────────────
ray.init()                →  spawns a pool of worker processes
                              (one per CPU core by default)  
add.remote(3, 7)          →  Ray scheduler picks an idle worker
                              worker deserializes args (3, 7)
                              worker runs add(3, 7)
                              worker stores result in Object Store  
ray.get(ref)              ←  main process fetches result from
                              Object Store

```

**A few important implications**:  
- Workers are separate processes, not threads. This means Ray bypasses Python's GIL entirely. True parallelism, not just concurrency. 
   
- Ray auto-detects your CPUs. Check how many workers Ray started.  
  
- The Object Store is shared memory. Results don't go back through your main process, they sit in a shared memory region (backed by /dev/shm on Linux/WSL) that all workers and your main process can read directly. This is why passing large objects is efficient.  
  
- Workers are reused. Ray doesn't spawn a new process per task. It maintains a warm pool and queues tasks to available workers.  
  
-  On a cluster, those workers could be on different machines entirely — same API, same code. That's the whole point of Ray.  

In [14]:
pprint(ray.cluster_resources())

{'CPU': 16.0,
 'GPU': 1.0,
 'accelerator_type:G': 1.0,
 'memory': 9373768500.0,
 'node:172.28.165.76': 1.0,
 'node:__internal_head__': 1.0,
 'object_store_memory': 4017329356.0}


In [10]:
@ray.remote
def where_am_i():
    return {
        "pid": os.getpid(),        # worker's process ID
        "hostname": os.uname().nodename
    }

In [11]:
# Submit 4 tasks — notice different PIDs (different workers)
refs = [where_am_i.remote() for _ in range(4)]
for r in ray.get(refs):
    print(r)

{'pid': 23809, 'hostname': 'mrinmoyAI'}
{'pid': 23804, 'hostname': 'mrinmoyAI'}
{'pid': 23807, 'hostname': 'mrinmoyAI'}
{'pid': 23816, 'hostname': 'mrinmoyAI'}


**Step 2 — Parallelism in action**  
This is the "aha" moment. Run both versions and compare the time:

In [19]:
# Normal Python function — for sequential baseline
def slow_square_normal(x):
    time.sleep(1)
    return x * x

# Ray task — for parallel version
@ray.remote
def slow_square_remote(x):
    time.sleep(1)   # simulate work
    return x * x

In [ ]:
# ── Sequential (normal Python) ──
start = time.time()
results = [slow_square_normal(i) for i in range(4)]
print(f"Sequential: {results} in {time.time()-start:.1f}s")
# → ~4.0 seconds

Sequential: [0, 1, 4, 9] in 4.0s


In [20]:
# ── Parallel (Ray) ──
start = time.time()
refs = [slow_square_remote.remote(i) for i in range(4)]
results = ray.get(refs)
print(f"Parallel:   {results} in {time.time()-start:.1f}s")
# → ~1.0 second

Parallel:   [0, 1, 4, 9] in 1.0s


What's happening:  

**Sequential**: each call waits 1s before the next starts → 4s total  
   
**Ray**: all 4 tasks are submitted instantly and run concurrently → ~1s total  
  
ray.get() also accepts a list of ObjectRefs and returns a list of results in the same order.

**Step 3 — ObjectRefs as first-class values**  
This is where Ray gets powerful. You can pass an ObjectRef as an argument to another task — Ray will automatically fetch the value before running the second task:

In [21]:
@ray.remote
def generate_data():
    return [1, 2, 3, 4, 5]

@ray.remote
def process_data(data):
    return sum(data)

In [22]:
# Chain tasks — no ray.get() in between!
data_ref = generate_data.remote()
result_ref = process_data.remote(data_ref)  # pass ref directly

print(ray.get(result_ref))  # 15

15


**Why this matters**: Ray resolves the dependency automatically. process_data won't start until generate_data finishes — Ray handles the synchronization for you. This is how you build task pipelines.

**Step 4 — ray.put() for large objects**  
If you have a large object (a big dataset, a model) that multiple tasks need, don't pass it by value — that serializes and sends it once per task. Instead, put it in the object store once.

In [23]:
@ray.remote
def process_chunk(data, i):
    return data[i] * 2

In [24]:
# Put large object in object store ONCE
big_data = list(range(1000))
data_ref = ray.put(big_data)   # stored once, shared across tasks

# All tasks reference the same object — no duplication
refs = [process_chunk.remote(data_ref, i) for i in range(10)]
print(ray.get(refs))

[0, 2, 4, 6, 8, 10, 12, 14, 16, 18]


### 2. Ray Actors  
#### What is a Ray Actor?  
A task is stateless — every call is independent, no memory between calls. An Actor is a stateful worker — it lives as a persistent process and remembers state across method calls.  
  
The mental model:  
```mermaid
Ray Task                        Ray Actor
────────────────                ──────────────────────────────
Stateless function              Stateful object (class instance)
Spawned per call                Lives until explicitly killed
No memory between calls         Remembers state across calls
Good for: data transforms       Good for: counters, model servers,
          batch processing                parameter stores, DB connections
```

**Step 1 — Your first Actor**

In [25]:
@ray.remote
class Counter:
    def __init__(self):
        self.count = 0

    def increment(self):
        self.count += 1

    def get(self):
        return self.count

In [26]:
# Create an actor instance — spawns a dedicated worker process
counter = Counter.remote()

# Call methods with .remote() — just like tasks
counter.increment.remote()
counter.increment.remote()
counter.increment.remote()

# Get the current state
print(ray.get(counter.get.remote()))  # 3

3


**Step 2 — Multiple independent Actors**  
Each actor instance is completely independent with its own state:

In [4]:
@ray.remote
class Counter:
    def __init__(self, name):
        self.name = name
        self.count = 0

    def increment(self, by=1):
        self.count += by

    def get(self):
        return f"{self.name}: {self.count}"

In [5]:
# Two completely independent actors — separate processes, separate state
counter_a = Counter.remote("A")
counter_b = Counter.remote("B")

counter_a.increment.remote()
counter_a.increment.remote()
counter_b.increment.remote(by=10)

print(ray.get(counter_a.get.remote()))  # A: 2
print(ray.get(counter_b.get.remote()))  # B: 10

A: 2
B: 10


**Step 3 — A realistic Actor: parameter store**  
A classic Ray pattern — an actor holding shared state that multiple tasks read from:

In [6]:
@ray.remote
class ParameterStore:
    def __init__(self):
        self.params = {}

    def set(self, key, value):
        self.params[key] = value

    def get(self, key):
        return self.params.get(key)

    def all(self):
        return self.params
    

@ray.remote
def worker(store, worker_id, key):
    # Each worker reads from the shared parameter store
    val = ray.get(store.get.remote(key))
    return f"Worker {worker_id} using {key}={val}"


In [7]:
# Create shared store
store = ParameterStore.remote()
store.set.remote("learning_rate", 0.001)
store.set.remote("batch_size", 32)

# Multiple workers all reading from same actor
refs = [worker.remote(store, i,  "learning_rate") for i in range(4)]
for r in ray.get(refs):
    print(r)

Worker 0 using learning_rate=0.001
Worker 1 using learning_rate=0.001
Worker 2 using learning_rate=0.001
Worker 3 using learning_rate=0.001


In [8]:
# Check final state
print(ray.get(store.all.remote()))

{'learning_rate': 0.001, 'batch_size': 32}


**Step 4 — Actor method calls are sequential**  
This is critical to understand. Method calls on a single actor are queued and executed one at a time — this is what guarantees state consistency:

In [9]:
@ray.remote
class OrderedWorker:
    def __init__(self):
        self.log = []

    def do_work(self, task_id):
        time.sleep(0.1)
        self.log.append(task_id)
        return task_id

    def get_log(self):
        return self.log

In [13]:
worker = OrderedWorker.remote()
# Submit 5 calls rapidly — they queue up on the actor
refs = [worker.do_work.remote(i) for i in range(5)]
print(ray.get(refs))

print(ray.get(worker.get_log.remote()))  # [0, 1, 2, 3, 4] — always in order

[0, 1, 2, 3, 4]
[0, 1, 2, 3, 4]


```mermaid
Ray Tasks                           Ray Actor Methods
─────────────────────────────────────────────────────────
Multiple tasks run in               Multiple method calls on the
PARALLEL across worker pool         SAME actor run SEQUENTIALLY

worker-0: task(0) ──►               actor process:
worker-1: task(1) ──►  all at       call(0) → call(1) → call(2) → call(3)
worker-2: task(2) ──►  same time    one at a time, in order
worker-3: task(3) ──►
```  
Why sequential for actors? Because the whole point of an actor is state safety. If two method calls ran simultaneously on the same actor, you'd get race conditions. So the full picture is:  
```mermaid
PARALLEL:  different tasks, different actors
SEQUENTIAL: methods on the SAME actor instance
```

**Tasks vs Actors — when to use which**  
```mermaid
Use a Task when:                    Use an Actor when:
────────────────                    ─────────────────
Pure function, no state             Need to maintain state
One-off computation                 Long-lived worker
Embarrassingly parallel             Sequential operations on shared state
Data transformation                 Model server, counter, cache
                                    Database connection holder
```

In [14]:
## Guess the output of the following
@ray.remote
class Counter:

    def __init__(self):
        self.count = 0

    def increment(self, by = 1):
        self.count += by
        
    def get(self):
        return self.count

In [15]:
c = Counter.remote()
refs = [c.increment.remote() for _ in range(5)]
print(ray.get(refs)) 

print(ray.get(c.get.remote())) # 5

[None, None, None, None, None]
5


### 3. Ray Object Store  
#### What is the Object Store?
It's a shared memory region that lives outside all worker processes. Every task, actor, and your main script can read from it without copying data between processes.  
```mermaid
┌─────────────────────────────────────────────────────┐
│  Ray Cluster                                        │
│                                                     │
│  ┌─────────────┐      ┌──────────────────────────┐  │
│  │ Main Process│      │      Object Store        │  │
│  └──────┬──────┘      │  (shared memory /dev/shm)│  │
│         │             │                          │  │
│  ┌──────┴──────┐      │  ref1 → [1,2,3,4,5]      │  │
│  │  Worker 0  │◄─────►│  ref2 → {"lr": 0.001}    │  │
│  ├────────────┤       │  ref3 → np.array(...)    │  │
│  │  Worker 1  │◄─────►│                          │  │
│  ├────────────┤       └──────────────────────────┘  │
│  │  Worker 2  │◄─────►        ▲                     │
│  └────────────┘               │                     │
│                        ray.put() / ray.get()        │
└─────────────────────────────────────────────────────┘
```

**Step 1 — Implicit vs Explicit Object Store**  
You've already been using the object store without knowing it. Every .remote() call result goes there automatically:

In [16]:
@ray.remote
def compute():
    return [1, 2, 3, 4]

# Implicit — Ray puts the result in object store automatically
ref = compute.remote()
result = ray.get(ref) # fetch it back from the object store
print(result) # [1, 2, 3, 4]

# Explicit — you put something directly
data = [1, 2, 3, 4, 5]
ref = ray.put(data) # manually store in object store
result = ray.get(ref) # fetch it back from the object store
print(result) # [1, 2, 3, 4, 5]

[1, 2, 3, 4]
[1, 2, 3, 4, 5]


**Step 2 — Why ray.put() matters for large objects**  
Without `ray.put()` — object is serialized and sent once per task:

In [3]:
@ray.remote
def process(
    data: np.ndarray|list[int], 
    i: int
) -> int:
    return data[i]*2

In [4]:
# Reproducible randomness (seed)
rng = np.random.default_rng(seed=42)   # modern way
arr = rng.integers(0, 10000, size=1000)

data_ref = ray.put(arr)
refs = [process.remote(data_ref, i) for i in range(4)]
print(ray.get(refs))

[np.int64(1784), np.int64(15478), np.int64(13090), np.int64(8776)]


**Step 3 — Object Store is zero-copy for numpy arrays**  
  
This is a big deal for ML workloads. When a worker reads a numpy array from the object store, Ray uses memory mapping — no copy is made at all:

In [5]:
@ray.remote
def inspect(data: Any):
    import ctypes
    return {
        "shape" : data.shape,
        "dtype": str(data.dtype),
        "is_writeable": data.flags.writeable
    }

In [ ]:
arr = np.ones((1000, 1000), dtype = np.float32)
ref = ray.put(arr)
pprint(ray.get(inspect.remote(ref)))

{'dtype': 'float32', 'is_writeable': False, 'shape': (1000, 1000)}


`is_writeable: False` tells you it's directly memory-mapped from the object store — zero copy. This is why Ray is fast for large array workloads.

**Step 4 — Object lifetime**  
  
Objects in the store are reference counted — they stay alive as long as something holds a reference:

In [9]:
data = list(range(1000))
ref = ray.put(data)

# Object lives as long as ref exists
result = ray.get(ref)
print(result[:5])   # [0, 1, 2, 3, 4]

# When ref goes out of scope or is deleted, object is evicted
del ref
# object store can now reclaim that memory

[0, 1, 2, 3, 4]


```mermaid
┌─────────────────────────────────────────────────────────┐
│                    Ray Primitives                       │
│                                                         │
│  @ray.remote def fn()  →  Task (stateless, parallel)    │
│  @ray.remote class C   →  Actor (stateful, sequential)  │
│  ray.put(obj)          →  Store in shared memory        │
│  ray.get(ref)          →  Fetch from shared memory      │
│  fn.remote()           →  Submit task → ObjectRef       │
│  C.remote()            →  Spawn actor process           │
│  actor.method.remote() →  Queue method call → ObjectRef │
└─────────────────────────────────────────────────────────┘
```